In [ ]:
from raw_file import RawFile
from Report import Report

In [ ]:
raw1 = RawFile('01_Kuraray_300TaN_CH052_WFT300E-WG-flat_CU4545F-300_Saesol-LPX-DP2-4-5lbf_1-5P_42-39V_in60s_r01.dat', 1, 193, 33.0)
raw2 = RawFile('02_Kuraray_300TaN_CH052_WFT300E-WG-flat_CU4545F-300_Saesol-LPX-DP2-4-5lbf_1-5P_63-61V_in60s_r01.dat', 2, 229, 22.1)
raw3 = RawFile('03_Kuraray_300TaN_CH052_WFT300E-WG-flat_CU4545F-300_Saesol-LPX-DP2-4-5lbf_1-5P_84-81V_in60s_r01.dat', 3, 247, 22.2)
raw4 = RawFile('04_Kuraray_300TaN_CH052_WFT300E-WG-flat_CU4545F-300_Saesol-LPX-DP2-4-5lbf_2P_42-39V_in60s_r01.dat',   4, 252, 23.9)
raw5 = RawFile('05_Kuraray_300TaN_CH052_WFT300E-WG-flat_CU4545F-300_Saesol-LPX-DP2-4-5lbf_2P_63-61V_in60s_r01.dat',   5, 329, 18.0)
raw6 = RawFile('06_Kuraray_300TaN_CH052_WFT300E-WG-flat_CU4545F-300_Saesol-LPX-DP2-4-5lbf_2P_84-81V_in60s_r01.dat',   6, 393, 17.4)

In [ ]:
report = Report()
report.add_file(raw1)
report.add_file(raw2)
report.add_file(raw3)
report.add_file(raw4)
report.add_file(raw5)
report.add_file(raw6)

In [ ]:
report.generate_report()

In [ ]:
from pptx import Presentation

# Path to your file
file_path = "araca_data_sample/Kuraray-Barrier CMP-12012025 FINAL SUBMITTED-sm.pptx"

prs = Presentation(file_path)

total_slides = len(prs.slides)
slides_with_title = 0
slides_without_title = []

for i, slide in enumerate(prs.slides):
    if slide.shapes.title is not None:
        slides_with_title += 1
    else:
        slides_without_title.append(i + 1)  # slide numbers start at 1

print(f"Total slides: {total_slides}")
print(f"Slides with title placeholder: {slides_with_title}")
print(f"Slides without title placeholder: {len(slides_without_title)}")

if slides_without_title:
    print("Slides missing title placeholder:")
    print(slides_without_title)

In [ ]:
file_path = "araca_data_sample/Kuraray-Barrier CMP-12012025 FINAL SUBMITTED-sm.pptx"

prs = Presentation(file_path)
prs.slides[0].shapes.title.text = "Hello"

In [ ]:
prs.save("out.pptx")

In [1]:
from dash import Dash, dcc, html, Input, Output
import plotly.graph_objects as go
from scipy import stats
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression
import numpy as np
import threading
import webbrowser
from raw_file import RawFile

# Load file
raw1 = RawFile('araca_data_sample/01_Kuraray_300TaN_CH052_WFT300E-WG-flat_CU4545F-300_Saesol-LPX-DP2-4-5lbf_1-5P_42-39V_in60s_r01.dat', 1, 193, 33.0)
rf = raw1

# Filter to steady-state interval only
start_idx = int(rf.interval[0] * rf.hz)
end_idx = int(rf.interval[1] * rf.hz)
df = rf.total_per_frame.iloc[start_idx:end_idx + 1].copy().reset_index(drop=True)

feature_columns = list(df.columns)

app = Dash(__name__)

app.layout = html.Div(
    [
        html.H1(
            "Feature Correlation Explorer",
            style={"textAlign": "center", "marginBottom": "20px"},
        ),
        html.Div(
            [
                html.Div(
                    [
                        html.Label("X-Axis Feature:", style={"fontWeight": "bold"}),
                        dcc.Dropdown(
                            id="x-feature",
                            options=[{"label": c, "value": c} for c in feature_columns],
                            value=feature_columns[0],
                            clearable=False,
                            style={"width": "100%"},
                        ),
                    ],
                    style={"width": "35%", "display": "inline-block", "verticalAlign": "top", "marginRight": "2%"},
                ),
                html.Div(
                    [
                        html.Label("Y-Axis Feature:", style={"fontWeight": "bold"}),
                        dcc.Dropdown(
                            id="y-feature",
                            options=[{"label": c, "value": c} for c in feature_columns],
                            value=feature_columns[1] if len(feature_columns) > 1 else feature_columns[0],
                            clearable=False,
                            style={"width": "100%"},
                        ),
                    ],
                    style={"width": "35%", "display": "inline-block", "verticalAlign": "top", "marginRight": "2%"},
                ),
                html.Div(
                    [
                        html.Label("Correlation Method:", style={"fontWeight": "bold"}),
                        dcc.RadioItems(
                            id="corr-method",
                            options=[
                                {"label": " Pearson", "value": "pearson"},
                                {"label": " Spearman", "value": "spearman"},
                            ],
                            value="pearson",
                            inline=True,
                            style={"marginTop": "8px"},
                        ),
                    ],
                    style={"width": "24%", "display": "inline-block", "verticalAlign": "top"},
                ),
            ],
            style={"padding": "10px 40px"},
        ),
        html.Div(
            id="corr-stats",
            style={"textAlign": "center", "fontSize": "18px", "margin": "15px 0", "fontWeight": "bold"},
        ),
        dcc.Graph(id="scatter-plot", style={"height": "70vh"}),
    ],
    style={"fontFamily": "Arial, sans-serif"},
)


@app.callback(
    Output("scatter-plot", "figure"),
    Output("corr-stats", "children"),
    Input("x-feature", "value"),
    Input("y-feature", "value"),
    Input("corr-method", "value"),
)
def update_plot(x_feat, y_feat, method):
    x = df[x_feat].replace([np.inf, -np.inf], np.nan).dropna()
    y = df[y_feat].replace([np.inf, -np.inf], np.nan).dropna()
    common_idx = x.index.intersection(y.index)
    x = x.loc[common_idx].values
    y = y.loc[common_idx].values

    if method == "pearson":
        corr, pval = stats.pearsonr(x, y)
        method_label = "Pearson"
    else:
        corr, pval = stats.spearmanr(x, y)
        method_label = "Spearman"

    max_points = 5000
    if len(x) > max_points:
        idx = np.random.choice(len(x), max_points, replace=False)
        idx.sort()
        x_plot, y_plot = x[idx], y[idx]
    else:
        x_plot, y_plot = x, y

    fig = go.Figure()
    fig.add_trace(
        go.Scattergl(
            x=x_plot, y=y_plot, mode="markers",
            marker=dict(size=3, opacity=0.5, color="royalblue"),
            name="Data",
        )
    )

    if len(x) > 1:
        if method == "pearson":
            # sklearn LinearRegression for Pearson
            lr = LinearRegression()
            lr.fit(x.reshape(-1, 1), y)
            x_line = np.array([x.min(), x.max()])
            y_line = lr.predict(x_line.reshape(-1, 1))
            fig.add_trace(
                go.Scatter(
                    x=x_line, y=y_line, mode="lines",
                    line=dict(color="red", width=2, dash="dash"),
                    name=f"Linear fit (y={lr.coef_[0]:.4g}x + {lr.intercept_:.4g})",
                )
            )
        else:
            # sklearn IsotonicRegression for Spearman (monotonic fit)
            # Direction matches the sign of Spearman correlation
            increasing = corr >= 0
            iso = IsotonicRegression(increasing=increasing, out_of_bounds='clip')
            sort_idx = np.argsort(x)
            x_sorted = x[sort_idx]
            y_sorted = y[sort_idx]
            y_iso = iso.fit_transform(x_sorted, y_sorted)
            # Deduplicate for cleaner line (isotonic produces step-like output)
            mask = np.diff(y_iso, prepend=y_iso[0] - 1) != 0
            mask[0] = True
            mask[-1] = True
            fig.add_trace(
                go.Scatter(
                    x=x_sorted[mask], y=y_iso[mask], mode="lines",
                    line=dict(color="red", width=2),
                    name=f"Isotonic fit ({'↑' if increasing else '↓'})",
                )
            )

    fig.update_layout(
        title=f"{x_feat}  vs  {y_feat}",
        xaxis_title=x_feat, yaxis_title=y_feat,
        template="plotly_white",
        legend=dict(x=0.01, y=0.99),
    )

    stats_text = f"{method_label} r = {corr:.6f}  |  p-value = {pval:.4e}  |  N = {len(x):,}"
    return fig, stats_text


print(f"Loaded file: {rf.file_basename}")
print(f"Steady-state interval: {rf.interval[0]}s – {rf.interval[1]}s")
print(f"Features: {feature_columns}")
print(f"Data points (steady-state): {len(df)}")

threading.Timer(1.5, lambda: webbrowser.open("http://127.0.0.1:8050")).start()
app.run(debug=False, port=8050)

/Users/gareginmazmanyan/Desktop/araca_data_automation/araca/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loaded file: 01_Kuraray_300TaN_CH052_WFT300E-WG-flat_CU4545F-300_Saesol-LPX-DP2-4-5lbf_1-5P_42-39V_in60s_r01.dat
Steady-state interval: 7s – 57s
Features: ['time (s)', 'Fz Total (lbf)', 'Fy Total (lbf)', '1 pound force = N', 'Fz Total (N)', 'Fy Total (N)', 'Pressure (Pa)', 'COF', 'Pad Rotation Rate (Rad/s)', 'Average Nominal Wafer Sliding Velocity (m/s)', 'v / P (m/Pas.s)', 'P.V (m.Pa/s)']
Data points (steady-state): 50001
